In [1]:

import torch
import yaml
from CHILI_centralAtoms import CHILI
from benchmark import validate_dataset_atom_count
from torch_geometric.loader import DataLoader
from test_vectordiff_overfitting import create_small_dataset
from mnist_ddpm_cond import train_vector_conditioned_ddpm

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
with open('configs/CHILI_mlp_config.yaml', "r") as file:
    config = yaml.safe_load(file)
dataset = CHILI(root=config["root"], dataset=config["dataset"], graph_type=config["graph_type"])

try:
    dataset.load_data_split(split_strategy = 'random')
except FileNotFoundError:
    print("No data split found, first run create_data_split")

print(dataset)


CHILI(3160)


In [4]:
# Create dataloaders
train_loader = DataLoader(
    dataset.train_set,
    batch_size=config["Train_config"]["batch_size"],
    shuffle=True,
)
val_loader = DataLoader(
    dataset.validation_set,
    batch_size=config["Train_config"]["batch_size"],
    shuffle=False,
)
test_loader = DataLoader(
    dataset.test_set,
    batch_size=config["Train_config"]["batch_size"],
    shuffle=False,
)

In [5]:


# Preprocess datasets for diffusion model
# Extract positions and xpdf from dataset
positions_list = []
xpdf_list = []

for data in train_loader:
    # Get absolute positions and reshape to [channels, height, width]
    # Assuming pos_abs has shape [num_atoms, 3]
    pos = data.pos_abs
    # Reshape to [1, 3, n, n] where n*n = num_atoms
    n = int(pos.shape[0] ** 0.5)  # Square root of number of atoms
    pos_reshaped = pos.view(config["Train_config"]["batch_size"], 3, 10, 10)
    positions_list.append(pos_reshaped)
    
    # Get xpdf values
    xpdf = data.y['xPDF']
    xpdf_list.append(xpdf[:,1,:])

# Stack to create batched tensors
positions_tensor = torch.cat(positions_list, dim=0)  # [batch, channels, height, width]
xpdf_tensor = torch.stack(xpdf_list)
xpdf_tensor = xpdf_tensor.squeeze(1)


RuntimeError: shape '[32, 3, 10, 10]' is invalid for input of size 4500

In [6]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=1000,
    learning_rate=1e-3,
    images=positions_tensor,
    conditioning_vectors=xpdf_tensor,
    epochs=100,
    batch_size=64,
    ema=True,
)

Using device: cuda:0


Training:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left

Training complete. Model saved to 'training_samples\T1000_lr0.001_epochs100_batch64_cond64\vector_conditioned_rgb_model.pt'


In [7]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=100,
    learning_rate=1e-3,
    images=positions_tensor,
    conditioning_vectors=xpdf_tensor,
    epochs=100,
    batch_size=64,
    ema=True,
)

Using device: cuda:0


Training:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left

Training complete. Model saved to 'training_samples\T100_lr0.001_epochs100_batch64_cond64\vector_conditioned_rgb_model.pt'


In [8]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=1000,
    learning_rate=1e-2,
    images=positions_tensor,
    conditioning_vectors=xpdf_tensor,
    epochs=100,
    batch_size=64,
    ema=True,
)

Using device: cuda:0


Training:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left

Training complete. Model saved to 'training_samples\T1000_lr0.01_epochs100_batch64_cond64\vector_conditioned_rgb_model.pt'


In [9]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=1000,
    learning_rate=1e-3,
    images=positions_tensor,
    conditioning_vectors=xpdf_tensor,
    epochs=300,
    batch_size=64,
    ema=True,
)

Using device: cuda:0


Training:   0%|          | 0/300 [00:00<?, ?it/s]

c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
c:\Users\toman\Documents\Ondro\master_thesis\mnist_ddpm_cond.py:725: UserWarning: Tight layout not applied. The left

Training complete. Model saved to 'training_samples\T1000_lr0.001_epochs300_batch64_cond64\vector_conditioned_rgb_model.pt'
